In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

print("Libraries Imported Successfully ✅")

Libraries Imported Successfully ✅


In [2]:
DATA_PATH = Path("../../Dataset/processed")

conn = sqlite3.connect("InsightX.db")

print("Database Connected Successfully ✅")

Database Connected Successfully ✅


In [3]:
orders = pd.read_csv(DATA_PATH / "orders_final_cleaned.csv")

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,Purchase Year,Purchase Month,Purchase Day,Purchase Weekday,Delivery Days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,2,Monday,8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,24,Tuesday,13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,8,Wednesday,9
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017,11,18,Saturday,13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018,2,13,Tuesday,2


In [4]:
orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

print("Orders Table Created Successfully ✅")

Orders Table Created Successfully ✅


In [5]:
query = """
SELECT COUNT(*) AS Total_Orders
FROM orders;
"""

pd.read_sql(query, conn)

,Total_Orders
0,98232


In [6]:
query = """
SELECT
    order_status,
    COUNT(*) AS Total
FROM orders
GROUP BY order_status
ORDER BY Total DESC;
"""

pd.read_sql(query, conn)

,order_status,Total
0,delivered,96471
1,shipped,604
2,unavailable,423
3,canceled,311
4,processing,212
5,invoiced,205
6,created,4
7,approved,2


In [7]:
query = """
SELECT
    `Purchase Year`,
    COUNT(*) AS Orders
FROM orders
GROUP BY `Purchase Year`
ORDER BY `Purchase Year`;
"""

pd.read_sql(query, conn)

,Purchase Year,Orders
0,2016,329
1,2017,44756
2,2018,53147


In [8]:
query = """
SELECT
    `Purchase Month`,
    COUNT(*) AS Orders
FROM orders
GROUP BY `Purchase Month`
ORDER BY `Purchase Month`;
"""

pd.read_sql(query, conn)

,Purchase Month,Orders
0,1,7963
1,2,8405
2,3,9766
3,4,9238
4,5,10472
5,6,9341
6,7,10175
7,8,10670
8,9,4243
9,10,4910


In [9]:
query = """
SELECT
ROUND(AVG(`Delivery Days`),2)
AS Avg_Delivery_Days
FROM orders;
"""

pd.read_sql(query, conn)

,Avg_Delivery_Days
0,15.75


In [10]:
query = """
SELECT
MAX(`Delivery Days`)
AS Maximum_Delivery
FROM orders;
"""

pd.read_sql(query, conn)

,Maximum_Delivery
0,692


In [11]:
query = """
SELECT
MIN(`Delivery Days`)
AS Minimum_Delivery
FROM orders;
"""

pd.read_sql(query, conn)

,Minimum_Delivery
0,0


In [12]:
query = """
SELECT
order_status,
ROUND(AVG(`Delivery Days`),2)
AS Avg_Delivery
FROM orders
GROUP BY order_status
ORDER BY Avg_Delivery;
"""

pd.read_sql(query, conn)

,order_status,Avg_Delivery
0,delivered,12.10
1,created,181.00
2,shipped,199.70
3,canceled,214.40
4,invoiced,222.21
5,unavailable,225.79
6,processing,235.50
7,approved,428.00


In [13]:
query = """
SELECT
`Purchase Month`,
ROUND(AVG(`Delivery Days`),2)
AS Avg_Delivery
FROM orders
GROUP BY `Purchase Month`
ORDER BY `Purchase Month`;
"""

pd.read_sql(query, conn)

,Purchase Month,Avg_Delivery
0,1,16.60
1,2,21.25
2,3,19.99
3,4,14.95
4,5,14.64
5,6,12.46
6,7,12.46
7,8,10.75
8,9,15.68
9,10,20.31


In [14]:
query = """
SELECT
`Purchase Weekday`,
COUNT(*) AS Orders
FROM orders
GROUP BY `Purchase Weekday`;
"""

pd.read_sql(query, conn)

,Purchase Weekday,Orders
0,Friday,13938
1,Monday,15987
2,Saturday,10754
3,Sunday,11843
4,Thursday,14578
5,Tuesday,15772
6,Wednesday,15360


In [15]:
query = """
SELECT

COUNT(*) AS Total_Orders,

ROUND(AVG(`Delivery Days`),2)
AS Avg_Delivery,

MAX(`Delivery Days`)
AS Max_Delivery,

MIN(`Delivery Days`)
AS Min_Delivery

FROM orders;
"""

pd.read_sql(query, conn)

,Total_Orders,Avg_Delivery,Max_Delivery,Min_Delivery
0,98232,15.75,692,0


In [16]:
query = """
SELECT
    order_status,
    COUNT(*) AS Total_Orders,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM orders),
        2
    ) AS Percentage
FROM orders
GROUP BY order_status
ORDER BY Total_Orders DESC;
"""

pd.read_sql(query, conn)

,order_status,Total_Orders,Percentage
0,delivered,96471,98.21
1,shipped,604,0.61
2,unavailable,423,0.43
3,canceled,311,0.32
4,processing,212,0.22
5,invoiced,205,0.21
6,created,4,0.00
7,approved,2,0.00


In [17]:
query = """
SELECT
    `Purchase Month`,
    COUNT(*) AS Total_Orders
FROM orders
GROUP BY `Purchase Month`
ORDER BY Total_Orders DESC;
"""

pd.read_sql(query, conn)

,Purchase Month,Total_Orders
0,8,10670
1,5,10472
2,7,10175
3,3,9766
4,6,9341
5,4,9238
6,2,8405
7,1,7963
8,11,7451
9,12,5598


In [18]:
query = """
SELECT
    `Purchase Month`,
    ROUND(AVG(`Delivery Days`),2) AS Avg_Delivery
FROM orders
GROUP BY `Purchase Month`
ORDER BY `Purchase Month`;
"""

pd.read_sql(query, conn)

,Purchase Month,Avg_Delivery
0,1,16.60
1,2,21.25
2,3,19.99
3,4,14.95
4,5,14.64
5,6,12.46
6,7,12.46
7,8,10.75
8,9,15.68
9,10,20.31


In [19]:
query = """
SELECT
    `Purchase Month`,
    MAX(`Delivery Days`) AS Maximum_Delivery
FROM orders
GROUP BY `Purchase Month`
ORDER BY Maximum_Delivery DESC;
"""

pd.read_sql(query, conn)

,Purchase Month,Maximum_Delivery
0,10,692
1,9,652
2,1,591
3,2,574
4,3,539
5,4,505
6,5,486
7,6,441
8,7,421
9,8,382


In [20]:
query = """
SELECT
    `Purchase Month`,
    MIN(`Delivery Days`) AS Minimum_Delivery
FROM orders
GROUP BY `Purchase Month`;
"""

pd.read_sql(query, conn)

,Purchase Month,Minimum_Delivery
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0
5,6,0
6,7,0
7,8,0
8,9,1
9,10,0


In [21]:
query = """
SELECT
    `Purchase Weekday`,
    ROUND(AVG(`Delivery Days`),2) AS Avg_Delivery
FROM orders
GROUP BY `Purchase Weekday`;
"""

pd.read_sql(query, conn)

,Purchase Weekday,Avg_Delivery
0,Friday,16.56
1,Monday,15.30
2,Saturday,16.74
3,Sunday,14.90
4,Thursday,15.90
5,Tuesday,15.08
6,Wednesday,15.97


In [22]:
query = """
SELECT
    `Purchase Weekday`,
    COUNT(*) AS Orders
FROM orders
GROUP BY `Purchase Weekday`
ORDER BY Orders DESC;
"""

pd.read_sql(query, conn)

,Purchase Weekday,Orders
0,Monday,15987
1,Tuesday,15772
2,Wednesday,15360
3,Thursday,14578
4,Friday,13938
5,Sunday,11843
6,Saturday,10754


In [23]:
query = """
SELECT
    order_id,
    order_status,
    `Delivery Days`
FROM orders
ORDER BY `Delivery Days` DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,order_id,order_status,Delivery Days
0,ddaec6fff982b13e7e048b627a11d6da,canceled,692
1,e04f1da1f48bf2bbffcf57b9824f76e1,invoiced,692
2,3c3ca08854ca922fe8e9cedfd6841c8a,unavailable,688
3,92d731517f17c26f16182d45498c4c3d,canceled,687
4,620b0acb9258b51defbc51804c5298d5,canceled,672
5,c4e980a1d822db426982878b3cfdda6e,invoiced,669
6,2e7a8482f6fb09756ca50c10d7bfc047,shipped,652
7,35b8e54d765e6b217e2dc5ab34f6b323,invoiced,642
8,bfd31c6f76ff82a41e7beb05565aec4d,canceled,640
9,c549f0d88f33bfe43351b893f1fb55ac,canceled,631


In [24]:
query = """
SELECT
    order_id,
    order_status,
    `Delivery Days`
FROM orders
ORDER BY `Delivery Days`
LIMIT 10;
"""

pd.read_sql(query, conn)

,order_id,order_status,Delivery Days
0,0760a852e4e9d89eb77bf631eaaf1c84,invoiced,0
1,38c1e3d4ed6a13cd0cf612d4c09766e9,delivered,0
2,d3ca7b82c922817b06e5ca21165c5ea2,delivered,0
3,bcd65d5c97ac448bb3efc90b759909fd,unavailable,0
4,f4dcd6f5120e9e46509a58be9e7462f2,shipped,0
5,1d893dd7ca5f77ebf5f59f0d2017eee0,delivered,0
6,21a8ffca665bc7a1087d31751a7b7cbc,delivered,0
7,f3c6775ba3d2d9fe2826f93b71f12008,delivered,0
8,434cecee7d1a65fc65358a632b6f725f,delivered,0
9,0538bda829ac14e64a201dc84fb1ba3b,shipped,0


In [25]:
query = """
SELECT
COUNT(*) AS Fast_Deliveries
FROM orders
WHERE `Delivery Days` < 5;
"""

pd.read_sql(query, conn)

,Fast_Deliveries
0,13458


In [26]:
query = """
SELECT
COUNT(*) AS Slow_Deliveries
FROM orders
WHERE `Delivery Days` > 15;
"""

pd.read_sql(query, conn)

,Slow_Deliveries
0,24867


In [27]:
query = """
SELECT
COUNT(*) AS Cancelled_Orders
FROM orders
WHERE order_status='canceled';
"""

pd.read_sql(query, conn)

,Cancelled_Orders
0,311


In [28]:
query = """
SELECT
COUNT(*) AS Delivered_Orders
FROM orders
WHERE order_status='delivered';
"""

pd.read_sql(query, conn)

,Delivered_Orders
0,96471


In [29]:
query = """
SELECT
DATE(order_purchase_timestamp) AS Purchase_Date,
COUNT(*) AS Orders
FROM orders
GROUP BY Purchase_Date
ORDER BY Orders DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,Purchase_Date,Orders
0,2017-11-24,1167
1,2017-11-25,496
2,2017-11-27,400
3,2017-11-26,389
4,2017-11-28,379
5,2018-05-07,365
6,2018-08-06,363
7,2018-05-14,359
8,2018-08-07,354
9,2018-05-16,353


In [30]:
query = """
SELECT

COUNT(*) AS Total_Orders,

SUM(
CASE
WHEN order_status='delivered'
THEN 1 ELSE 0
END
) AS Delivered,

SUM(
CASE
WHEN order_status='canceled'
THEN 1 ELSE 0
END
) AS Cancelled,

ROUND(AVG(`Delivery Days`),2)
AS Avg_Delivery,

MAX(`Delivery Days`)
AS Max_Delivery,

MIN(`Delivery Days`)
AS Min_Delivery

FROM orders;
"""

pd.read_sql(query, conn)

,Total_Orders,Delivered,Cancelled,Avg_Delivery,Max_Delivery,Min_Delivery
0,98232,96471,311,15.75,692,0


In [6]:
import pandas as pd

In [7]:
print(pd.__version__)

2.3.3


In [9]:
from pathlib import Path

DATA_PATH = Path("../../Dataset/processed")

In [10]:
customers = pd.read_csv(DATA_PATH / "customers.csv")
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [12]:
import sqlite3

conn = sqlite3.connect(":memory:")

print("SQLite Connected ✅")

SQLite Connected ✅


In [13]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Customers table created successfully ✅")

Customers table created successfully ✅


In [14]:
query = """
SELECT *
FROM customers
LIMIT 5;
"""

pd.read_sql(query, conn)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [15]:
query = """
SELECT
    COUNT(*) AS Total_Customers
FROM customers;
"""

pd.read_sql(query, conn)

,Total_Customers
0,99441


In [16]:
query = """
SELECT
    COUNT(DISTINCT customer_unique_id) AS Unique_Customers
FROM customers;
"""

pd.read_sql(query, conn)

,Unique_Customers
0,96096


In [17]:
query = """
SELECT
    COUNT(DISTINCT customer_city) AS Total_Cities
FROM customers;
"""

pd.read_sql(query, conn)

,Total_Cities
0,4119


In [18]:
query = """
SELECT
    COUNT(DISTINCT customer_state) AS Total_States
FROM customers;
"""

pd.read_sql(query, conn)

,Total_States
0,27


In [19]:
query = """
SELECT
    customer_state,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_state
ORDER BY Total_Customers DESC;
"""

pd.read_sql(query, conn)

,customer_state,Total_Customers
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


In [20]:
query = """
SELECT
    customer_city,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_city
ORDER BY Total_Customers DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_city,Total_Customers
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [21]:
query = """
SELECT
    customer_state,
    COUNT(*) AS Total_Customers,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM customers),
        2
    ) AS Percentage
FROM customers
GROUP BY customer_state
ORDER BY Total_Customers DESC;
"""

pd.read_sql(query, conn)

,customer_state,Total_Customers,Percentage
0,SP,41746,41.98
1,RJ,12852,12.92
2,MG,11635,11.70
3,RS,5466,5.50
4,PR,5045,5.07
5,SC,3637,3.66
6,BA,3380,3.40
7,DF,2140,2.15
8,ES,2033,2.04
9,GO,2020,2.03


In [22]:
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_city) AS Total_Cities
FROM customers
GROUP BY customer_state
ORDER BY Total_Cities DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_state,Total_Cities
0,MG,745
1,SP,629
2,RS,379
3,PR,364
4,BA,353
5,SC,240
6,GO,178
7,CE,161
8,PE,152
9,RJ,149


In [23]:
query = """
SELECT
    customer_state,
    ROUND(
        COUNT(*) * 1.0 /
        COUNT(DISTINCT customer_city),
        2
    ) AS Avg_Customers_Per_City
FROM customers
GROUP BY customer_state
ORDER BY Avg_Customers_Per_City DESC;
"""

pd.read_sql(query, conn)

,customer_state,Avg_Customers_Per_City
0,DF,356.67
1,RJ,86.26
2,SP,66.37
3,AM,29.60
4,RR,23.00
5,ES,21.40
6,MG,15.62
7,SC,15.15
8,RS,14.42
9,PR,13.86


In [24]:
query = """
SELECT
    customer_state,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_state
HAVING COUNT(*) > 1000
ORDER BY Total_Customers DESC;
"""

pd.read_sql(query, conn)

,customer_state,Total_Customers
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


In [25]:
query = """
SELECT
    customer_city,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_city
HAVING COUNT(*) > 500
ORDER BY Total_Customers DESC;
"""

pd.read_sql(query, conn)

,customer_city,Total_Customers
0,sao paulo,15540
1,rio de janeiro,6882
2,belo horizonte,2773
3,brasilia,2131
4,curitiba,1521
5,campinas,1444
6,porto alegre,1379
7,salvador,1245
8,guarulhos,1189
9,sao bernardo do campo,938


In [26]:
query = """
SELECT
    customer_zip_code_prefix,
    COUNT(*) AS Customers
FROM customers
GROUP BY customer_zip_code_prefix
ORDER BY Customers DESC
LIMIT 20;
"""

pd.read_sql(query, conn)

,customer_zip_code_prefix,Customers
0,22790,142
1,24220,124
2,22793,121
3,24230,117
4,22775,110
5,29101,101
6,13212,95
7,35162,93
8,22631,89
9,38400,87


In [27]:
query = """
SELECT
    customer_city,
    customer_state,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_city, customer_state
ORDER BY Total_Customers DESC
LIMIT 20;
"""

pd.read_sql(query, conn)

,customer_city,customer_state,Total_Customers
0,sao paulo,SP,15540
1,rio de janeiro,RJ,6882
2,belo horizonte,MG,2773
3,brasilia,DF,2131
4,curitiba,PR,1521
5,campinas,SP,1444
6,porto alegre,RS,1379
7,salvador,BA,1245
8,guarulhos,SP,1189
9,sao bernardo do campo,SP,938


In [28]:
query = """
SELECT
    customer_city,
    customer_state,
    COUNT(*) AS Total_Customers
FROM customers
GROUP BY customer_city, customer_state
ORDER BY Total_Customers ASC
LIMIT 20;
"""

pd.read_sql(query, conn)

,customer_city,customer_state,Total_Customers
0,abadiania,GO,1
1,abdon batista,SC,1
2,acajutiba,BA,1
3,acari,RN,1
4,acucena,MG,1
5,adhemar de barros,PR,1
6,adrianopolis,PR,1
7,adustina,BA,1
8,agisse,SP,1
9,agrestina,PE,1


In [29]:
query = """
SELECT
    customer_state,
    COUNT(*) AS Total_Customers,
    DENSE_RANK() OVER(
        ORDER BY COUNT(*) DESC
    ) AS State_Rank
FROM customers
GROUP BY customer_state;
"""

pd.read_sql(query, conn)

,customer_state,Total_Customers,State_Rank
0,SP,41746,1
1,RJ,12852,2
2,MG,11635,3
3,RS,5466,4
4,PR,5045,5
5,SC,3637,6
6,BA,3380,7
7,DF,2140,8
8,ES,2033,9
9,GO,2020,10


In [30]:
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_city) AS Total_Cities
FROM customers
GROUP BY customer_state
ORDER BY Total_Cities DESC;
"""

pd.read_sql(query, conn)

,customer_state,Total_Cities
0,MG,745
1,SP,629
2,RS,379
3,PR,364
4,BA,353
5,SC,240
6,GO,178
7,CE,161
8,PE,152
9,RJ,149


In [31]:
query = """
SELECT
    customer_state,
    ROUND(AVG(customer_zip_code_prefix),0)
    AS Avg_Zip_Code
FROM customers
GROUP BY customer_state
ORDER BY Avg_Zip_Code;
"""

pd.read_sql(query, conn)

,customer_state,Avg_Zip_Code
0,SP,9250.0
1,RJ,23965.0
2,ES,29297.0
3,MG,35211.0
4,BA,44102.0
5,SE,49193.0
6,PE,53675.0
7,AL,57216.0
8,PB,58273.0
9,RN,59288.0


In [32]:
query = """
SELECT
COUNT(*) AS Customers_SP
FROM customers
WHERE customer_state='SP';
"""

pd.read_sql(query, conn)

,Customers_SP
0,41746


In [33]:
query = """
SELECT
customer_state,
COUNT(*) AS Customers,
ROUND(
COUNT(*)*100.0/
(SELECT COUNT(*) FROM customers),2
) AS Share_Percentage
FROM customers
GROUP BY customer_state
ORDER BY Customers DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,customer_state,Customers,Share_Percentage
0,SP,41746,41.98
1,RJ,12852,12.92
2,MG,11635,11.70
3,RS,5466,5.50
4,PR,5045,5.07


In [34]:
query = """
SELECT

COUNT(*) AS Total_Customers,

COUNT(DISTINCT customer_unique_id)
AS Unique_Customers,

COUNT(DISTINCT customer_state)
AS States,

COUNT(DISTINCT customer_city)
AS Cities

FROM customers;
"""

pd.read_sql(query, conn)

,Total_Customers,Unique_Customers,States,Cities
0,99441,96096,27,4119


In [35]:
products = pd.read_csv(DATA_PATH / "products.csv")

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [36]:
products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Products table created successfully!")

✅ Products table created successfully!


In [37]:
query = """
SELECT *
FROM products
LIMIT 5;
"""

pd.read_sql(query, conn)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [38]:
query = """
SELECT
COUNT(*) AS Total_Products
FROM products;
"""

pd.read_sql(query, conn)

,Total_Products
0,32951


In [39]:
query = """
SELECT
COUNT(DISTINCT product_category_name)
AS Total_Categories
FROM products;
"""

pd.read_sql(query, conn)

,Total_Categories
0,74


In [40]:
query = """
SELECT
COUNT(*) AS Missing_Category
FROM products
WHERE product_category_name IS NULL;
"""

pd.read_sql(query, conn)

,Missing_Category
0,0


In [41]:
query = """
SELECT
product_category_name,
COUNT(*) AS Total_Products
FROM products
GROUP BY product_category_name
ORDER BY Total_Products DESC;
"""

pd.read_sql(query, conn)

,product_category_name,Total_Products
0,cama_mesa_banho,3029
1,esporte_lazer,2867
2,moveis_decoracao,2657
3,beleza_saude,2444
4,utilidades_domesticas,2335
...,...,...
69,fashion_roupa_infanto_juvenil,5
70,casa_conforto_2,5
71,pc_gamer,3
72,seguros_e_servicos,2


In [42]:
query = """
SELECT
product_category_name,
COUNT(*) AS Total_Products
FROM products
GROUP BY product_category_name
ORDER BY Total_Products DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_category_name,Total_Products
0,cama_mesa_banho,3029
1,esporte_lazer,2867
2,moveis_decoracao,2657
3,beleza_saude,2444
4,utilidades_domesticas,2335
5,automotivo,1900
6,informatica_acessorios,1639
7,brinquedos,1411
8,relogios_presentes,1329
9,telefonia,1134


In [43]:
query = """
SELECT
product_category_name,
COUNT(*) AS Total_Products
FROM products
GROUP BY product_category_name
ORDER BY Total_Products ASC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_category_name,Total_Products
0,cds_dvds_musicais,1
1,seguros_e_servicos,2
2,pc_gamer,3
3,casa_conforto_2,5
4,fashion_roupa_infanto_juvenil,5
5,tablets_impressao_imagem,9
6,la_cuisine,10
7,moveis_colchao_e_estofado,10
8,portateis_cozinha_e_preparadores_de_alimentos,10
9,fraldas_higiene,12


In [44]:
query = """
SELECT
ROUND(AVG(product_weight_g),2)
AS Avg_Product_Weight
FROM products;
"""

pd.read_sql(query, conn)

,Avg_Product_Weight
0,2276.47


In [45]:
query = """
SELECT
product_id,
product_category_name,
product_weight_g
FROM products
ORDER BY product_weight_g DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_id,product_category_name,product_weight_g
0,26644690fde745fc4654719c3904e1db,cama_mesa_banho,40425.0
1,d0877f0094337c414d23f5a3c7bad20c,moveis_escritorio,30000.0
2,53f92b0474f91fcb5bd188c6a8075c38,utilidades_domesticas,30000.0
3,ceeba7d5636e59173cc5f484e913db3d,Unknown,30000.0
4,f97ad9066c718a6cef93dfcf253d3e0d,moveis_decoracao,30000.0
5,363a9f5b97bf194da23858be722a7aa5,construcao_ferramentas_construcao,30000.0
6,dcfeedf441c38e5e7e58ffce194af2bb,beleza_saude,30000.0
7,1c57458e824ca3d974ec1831a1a55e72,pet_shop,30000.0
8,c04e948c6900ce99ac47d89b3b6d70cd,cool_stuff,30000.0
9,0a859d8dc68f6a746b4709217110c439,esporte_lazer,30000.0


In [46]:
query = """
SELECT
product_id,
product_category_name,
product_weight_g
FROM products
WHERE product_weight_g IS NOT NULL
ORDER BY product_weight_g ASC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_id,product_category_name,product_weight_g
0,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0
1,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0
2,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0
3,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0
4,ad7d07f5775feab3f20504d1ad3fff11,moveis_decoracao,2.0
5,7ddb76f2c7237acc852358b95e7946a8,papelaria,2.0
6,f9fafac43d3416d92ecc303fdeb1743d,moveis_decoracao,2.0
7,8aae4df46baf1278422b69edbb50bd35,moveis_decoracao,2.0
8,5837bba0ce6e35e6f2dc5c3e223e3276,moveis_decoracao,2.0
9,632921e8d6c6b23ae47ff20950a56854,cama_mesa_banho,25.0


In [47]:
query = """
SELECT

MIN(product_weight_g) AS Min_Weight,

ROUND(AVG(product_weight_g),2)
AS Avg_Weight,

MAX(product_weight_g) AS Max_Weight

FROM products;
"""

pd.read_sql(query, conn)

,Min_Weight,Avg_Weight,Max_Weight
0,0.0,2276.47,40425.0


In [48]:
query = """
SELECT
ROUND(AVG(LENGTH(product_category_name)),2)
AS Avg_Category_Name_Length
FROM products;
"""

pd.read_sql(query, conn)

,Avg_Category_Name_Length
0,14.81


In [49]:
query = """
SELECT
COUNT(*) AS Products_Without_Category
FROM products
WHERE product_category_name IS NULL;
"""

pd.read_sql(query, conn)

,Products_Without_Category
0,0


In [50]:
query = """
SELECT
ROUND(AVG(product_description_lenght),2)
AS Avg_Description_Length
FROM products;
"""

pd.read_sql(query, conn)

,Avg_Description_Length
0,768.23


In [51]:
query = """
SELECT
ROUND(AVG(product_name_lenght),2)
AS Avg_Product_Name_Length
FROM products;
"""

pd.read_sql(query, conn)

,Avg_Product_Name_Length
0,48.52


In [52]:
query = """
SELECT
ROUND(AVG(product_photos_qty),2)
AS Avg_Photos
FROM products;
"""

pd.read_sql(query, conn)

,Avg_Photos
0,2.17


In [53]:
query = """
SELECT
COUNT(*) AS Products_With_5Plus_Photos
FROM products
WHERE product_photos_qty > 5;
"""

pd.read_sql(query, conn)

,Products_With_5Plus_Photos
0,1817


In [54]:
query = """
SELECT

ROUND(AVG(product_length_cm),2) AS Avg_Length,

ROUND(AVG(product_height_cm),2) AS Avg_Height,

ROUND(AVG(product_width_cm),2) AS Avg_Width

FROM products;
"""

pd.read_sql(query, conn)

,Avg_Length,Avg_Height,Avg_Width
0,30.82,16.94,23.2


In [55]:
query = """
SELECT

product_id,

product_category_name,

(product_length_cm *
 product_height_cm *
 product_width_cm) AS Volume

FROM products

ORDER BY Volume DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,product_id,product_category_name,Volume
0,256a9c364b75753b97bee410c9491ad8,utilidades_domesticas,296208.0
1,0b48eade13cfad433122f23739a66898,moveis_decoracao,294000.0
2,f227e2d44f10f7dad30fb4dfa839e7a2,moveis_sala,294000.0
3,3eb14e65e4208c6d94b7a32e41add538,moveis_sala,294000.0
4,c1e0531cb1864fd3a0cae57dca55ca80,moveis_sala,294000.0
5,90c1b4e040d1d1c45897ec2dad4a809d,moveis_sala,293706.0
6,c6fdec160d0f8f488d9041316c85051d,moveis_sala,288000.0
7,8d6f2c3454002d3f5aa7479a7fad7794,moveis_sala,288000.0
8,99ff40856c47a638df807c0a144470cc,moveis_sala,288000.0
9,0e9dfb804bafa3d68ef3ee7a621abfb2,moveis_sala,287980.0


In [56]:
query = """
SELECT

product_id,

product_category_name,

(product_length_cm *
 product_height_cm *
 product_width_cm) AS Volume

FROM products

WHERE product_length_cm IS NOT NULL

ORDER BY Volume ASC

LIMIT 10;
"""

pd.read_sql(query, conn)

,product_id,product_category_name,Volume
0,106392145fca363410d287a815be6de4,cama_mesa_banho,168.0
1,2f763ba79d9cd987b2034aac7ceffe06,eletronicos,288.0
2,3bb7f144022e6732727d8d838a7b13b3,esporte_lazer,352.0
3,723f62c36c298037716bf60c0ad0df3a,relogios_presentes,352.0
4,55ec10b39713ee81ee2c5bcb558cf866,fashion_bolsas_e_acessorios,352.0
5,acd427ee119d5c71c7a85ae488cb0a6a,fashion_bolsas_e_acessorios,352.0
6,df473738565b52f77b4e22b328b41576,construcao_ferramentas_ferramentas,352.0
7,43724b27731595d954e911f443fb1cc4,construcao_ferramentas_construcao,352.0
8,5e21d5cab5d33e770d8150a4ee6117db,relogios_presentes,352.0
9,42f14eda412436b96cde4f4025ed5f25,esporte_lazer,352.0


In [57]:
query = """
SELECT

COUNT(*) AS Total_Products,

COUNT(DISTINCT product_category_name)
AS Categories,

ROUND(AVG(product_weight_g),2)
AS Avg_Weight,

ROUND(AVG(product_photos_qty),2)
AS Avg_Photos

FROM products;
"""

pd.read_sql(query, conn)

,Total_Products,Categories,Avg_Weight,Avg_Photos
0,32951,74,2276.47,2.17


In [58]:
order_items = pd.read_csv(DATA_PATH / "order_items.csv")

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Order Items Loaded")

✅ Order Items Loaded


In [59]:
payments = pd.read_csv(DATA_PATH / "payments.csv")

payments.to_sql(
    "payments",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Payments Loaded")

✅ Payments Loaded


In [60]:
reviews = pd.read_csv(DATA_PATH / "reviews.csv")

reviews.to_sql(
    "reviews",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Reviews Loaded")

✅ Reviews Loaded


In [61]:
sellers = pd.read_csv(DATA_PATH / "sellers.csv")

sellers.to_sql(
    "sellers",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Sellers Loaded")

✅ Sellers Loaded


In [62]:
category_translation = pd.read_csv(
    DATA_PATH / "category_translation.csv"
)

category_translation.to_sql(
    "category_translation",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Category Translation Loaded")

✅ Category Translation Loaded


In [63]:
geolocation = pd.read_csv(DATA_PATH / "geolocation.csv")

geolocation.to_sql(
    "geolocation",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Geolocation Loaded")

✅ Geolocation Loaded


In [64]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,customers
1,products
2,order_items
3,payments
4,reviews
5,sellers
6,category_translation
7,geolocation


In [65]:
query = """
SELECT
ROUND(SUM(price),2) AS Total_Revenue
FROM order_items;
"""

pd.read_sql(query, conn)

,Total_Revenue
0,13591643.7


In [66]:
query = """
SELECT
ROUND(SUM(freight_value),2) AS Total_Freight
FROM order_items;
"""

pd.read_sql(query, conn)

,Total_Freight
0,2251909.54


In [67]:
query = """
SELECT
ROUND(
SUM(price + freight_value),2
) AS Gross_Revenue
FROM order_items;
"""

pd.read_sql(query, conn)

,Gross_Revenue
0,15843553.24


In [68]:
query = """
SELECT

ROUND(
SUM(price)/
COUNT(DISTINCT order_id),
2
) AS Average_Order_Value

FROM order_items;
"""

pd.read_sql(query, conn)

,Average_Order_Value
0,137.75


In [69]:
query = """
SELECT

COUNT(DISTINCT order_id)
AS Total_Orders

FROM order_items;
"""

pd.read_sql(query, conn)

,Total_Orders
0,98666


In [70]:
query = """
SELECT

COUNT(product_id)
AS Products_Sold

FROM order_items;
"""

pd.read_sql(query, conn)

,Products_Sold
0,112650


In [71]:
query = """
SELECT

seller_id,

ROUND(SUM(price),2)
AS Revenue

FROM order_items

GROUP BY seller_id

ORDER BY Revenue DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,seller_id,Revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55
9,955fee9216a65b617aa5c0531780ce60,135171.70


In [72]:
query = """
SELECT

product_id,

ROUND(SUM(price),2)
AS Revenue

FROM order_items

GROUP BY product_id

ORDER BY Revenue DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,product_id,Revenue
0,bb50f2e236e5eea0100680137654686c,63885.00
1,6cdd53843498f92890544667809f1595,54730.20
2,d6160fb7873f184099d9bc95e30376af,48899.34
3,d1c427060a0f73f6b889a5c7c61f2ac4,47214.51
4,99a4788cb24856965c36a24e339b6058,43025.56
5,3dd2a17168ec895c781a9191c1e95ad7,41082.60
6,25c38557cf793876c5abdd5931f922db,38907.32
7,5f504b3a1c75b73d6151be81eb05bdc9,37733.90
8,53b36df67ebb7c41585e8d54d6772e08,37683.42
9,aca2eb7d00ea1a7b8ebd4e68314663af,37608.90


In [73]:
query = """
SELECT

order_id,

ROUND(SUM(price),2)
AS Order_Value

FROM order_items

GROUP BY order_id

ORDER BY Order_Value DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,order_id,Order_Value
0,03caa2c082116e1d31e67e9ae3700499,13440.00
1,736e1922ae60d0d6a89247b851902527,7160.00
2,0812eb902a67711a1cb742b3cdaa65ae,6735.00
3,fefacc66af859508bf1a7934eab1e97f,6729.00
4,f5136e38d1a14a4dbd87dff67da82701,6499.00
5,2cc9089445046817a7539d90805e6e5a,5934.60
6,a96610ab360d42a2e5335a3998b4718a,4799.00
7,199af31afc78c699f0dbf71fb178d4d4,4690.00
8,b4c4b76c642808cbe472a32b86cddc95,4599.90
9,8dbc85d1447242f3b127dda390d56e19,4590.00


In [74]:
query = """
SELECT

order_id,

ROUND(SUM(price),2)
AS Order_Value

FROM order_items

GROUP BY order_id

ORDER BY Order_Value ASC

LIMIT 20;
"""

pd.read_sql(query, conn)

,order_id,Order_Value
0,3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85
1,6e864b3f0ec71031117ad4cf46b7f2a1,0.85
2,f1d5c2e6867fa93ceee9ef9b34a53cbf,2.20
3,e8bbc1d69fee39eee4c72cb5c969e39d,2.29
4,38bcb524e1c38c2c1b60600a80fc8999,2.90
5,de03f4f4bb610147a9bbfbed56438cbb,2.99
6,f09e36e258656850b92657ac5f67b6d5,3.00
7,f9ccaff7267fd0cf076e795b1fae8b69,3.00
8,84d54e2fe6fc76e34d897b32c59dd75d,3.49
9,37193e64eb9a46b7f3197762f242b20a,3.50


In [76]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,customers
1,products
2,order_items
3,payments
4,reviews
5,sellers
6,category_translation
7,geolocation


In [77]:
orders = pd.read_csv(DATA_PATH / "orders_final_cleaned.csv")

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

print("✅ Orders Loaded Successfully")

✅ Orders Loaded Successfully


In [78]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,customers
1,products
2,order_items
3,payments
4,reviews
5,sellers
6,category_translation
7,geolocation
8,orders


In [79]:
query = """
SELECT COUNT(*)
FROM orders;
"""

pd.read_sql(query, conn)

,COUNT(*)
0,98232


In [82]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name


In [84]:
conn = sqlite3.connect("../../Dataset/processed/insightx.db")

In [85]:
import sqlite3
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../Dataset/processed")

conn = sqlite3.connect("../../Dataset/processed/insightx.db")

datasets = {
    "orders": "orders_final_cleaned.csv",
    "customers": "customers.csv",
    "products": "products.csv",
    "order_items": "order_items.csv",
    "payments": "payments.csv",
    "reviews": "reviews.csv",
    "sellers": "sellers.csv",
    "category_translation": "category_translation.csv",
    "geolocation": "geolocation.csv"
}

for table, file in datasets.items():
    df = pd.read_csv(DATA_PATH / file)
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"✅ {table} loaded")

print("\n🎉 All tables loaded successfully!")

✅ orders loaded
✅ customers loaded
✅ products loaded
✅ order_items loaded
✅ payments loaded
✅ reviews loaded
✅ sellers loaded
✅ category_translation loaded
✅ geolocation loaded

🎉 All tables loaded successfully!


In [86]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql(query, conn)

,name
0,orders
1,customers
2,products
3,order_items
4,payments
5,reviews
6,sellers
7,category_translation
8,geolocation


In [87]:
query = """
SELECT
    COALESCE(ct.product_category_name_english,
             p.product_category_name) AS Category,

    COUNT(DISTINCT oi.order_id) AS Total_Orders,

    COUNT(oi.product_id) AS Products_Sold,

    ROUND(SUM(oi.price),2) AS Revenue,

    ROUND(AVG(oi.price),2) AS Avg_Product_Price

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY Category

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,Category,Total_Orders,Products_Sold,Revenue,Avg_Product_Price
0,health_beauty,8836,9670,1258681.34,130.16
1,watches_gifts,5624,5991,1205005.68,201.14
2,bed_bath_table,9417,11115,1036988.68,93.30
3,sports_leisure,7720,8641,988048.97,114.34
4,computers_accessories,6689,7827,911954.32,116.51
...,...,...,...,...,...
69,flowers,29,33,1110.04,33.64
70,home_comfort_2,24,30,760.27,25.34
71,cds_dvds_musicals,12,14,730.00,52.14
72,fashion_childrens_clothes,8,8,569.85,71.23


In [88]:
query = """
SELECT

c.customer_unique_id,

COUNT(DISTINCT o.order_id) AS Orders,

ROUND(SUM(oi.price),2) AS Total_Spent,

ROUND(AVG(oi.price),2) AS Avg_Order_Value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY c.customer_unique_id

ORDER BY Total_Spent DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,customer_unique_id,Orders,Total_Spent,Avg_Order_Value
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,1680.00
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,3694.00
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,1790.00
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,6735.00
4,459bef486812aa25204be022145caa62,1,6729.00,6729.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,6499.00
6,4007669dec559734d6f53e029e360987,1,5934.60,989.10
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,4690.00
8,48e1ac109decbb87765a3eade6854098,1,4590.00,4590.00
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.00,2200.00


In [89]:
query = """
SELECT

s.seller_state,

COUNT(DISTINCT s.seller_id) AS Sellers,

ROUND(SUM(oi.price),2) AS Revenue

FROM sellers s

JOIN order_items oi
ON s.seller_id = oi.seller_id

GROUP BY s.seller_state

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,seller_state,Sellers,Revenue
0,SP,1849,8753396.21
1,PR,349,1261887.21
2,MG,244,1011564.74
3,RJ,171,843984.22
4,SC,190,632426.07
5,RS,129,378559.54
6,BA,19,285561.56
7,DF,30,97749.48
8,PE,9,91493.85
9,GO,40,66399.21


In [90]:
query = """
SELECT

c.customer_state,

COUNT(DISTINCT o.order_id) AS Orders,

ROUND(SUM(oi.price),2) AS Revenue,

ROUND(AVG(oi.price),2) AS Avg_Order_Value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY c.customer_state

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,customer_state,Orders,Revenue,Avg_Order_Value
0,SP,40969,5148312.82,109.56
1,RJ,12577,1796125.28,124.88
2,MG,11476,1572099.91,120.38
3,RS,5406,739270.71,119.08
4,PR,4973,677650.82,118.59
5,SC,3594,518843.39,124.90
6,BA,3315,502624.06,134.07
7,DF,2103,299909.74,125.91
8,GO,1989,287525.85,124.42
9,ES,2011,272563.44,121.57


In [91]:
query = """
SELECT

strftime('%Y-%m',
o.order_purchase_timestamp) AS Month,

ROUND(SUM(oi.price),2) AS Revenue,

COUNT(DISTINCT o.order_id) AS Orders

FROM orders o

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY Month

ORDER BY Month;
"""

pd.read_sql(query, conn)

,Month,Revenue,Orders
0,2016-09,267.36,3
1,2016-10,49507.66,308
2,2016-12,10.90,1
3,2017-01,120312.87,789
4,2017-02,247303.02,1733
5,2017-03,374229.40,2639
6,2017-04,359229.62,2386
7,2017-05,505185.88,3650
8,2017-06,432574.42,3211
9,2017-07,494737.64,3949


In [92]:
query = """
SELECT

c.customer_city,

COUNT(DISTINCT o.order_id) AS Orders,

ROUND(SUM(oi.price),2) AS Revenue,

ROUND(AVG(oi.price),2) AS Avg_Order_Value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY c.customer_city

ORDER BY Revenue DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,customer_city,Orders,Revenue,Avg_Order_Value
0,sao paulo,15231,1892017.29,107.44
1,rio de janeiro,6726,975166.85,126.25
2,belo horizonte,2728,350976.61,112.46
3,brasilia,2094,299226.05,126.36
4,curitiba,1503,208330.06,119.46
5,porto alegre,1362,187428.36,117.07
6,campinas,1419,186512.26,113.52
7,salvador,1218,177739.58,127.87
8,guarulhos,1157,141396.26,108.18
9,niteroi,838,117356.62,120.74


In [93]:
query = """
SELECT

p.product_id,

COALESCE(ct.product_category_name_english,
p.product_category_name) AS Category,

COUNT(*) AS Units_Sold,

ROUND(SUM(oi.price),2) AS Revenue

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY p.product_id

ORDER BY Units_Sold DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,product_id,Category,Units_Sold,Revenue
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,37608.90
1,99a4788cb24856965c36a24e339b6058,bed_bath_table,488,43025.56
2,422879e10f46682990de24d770e7f83d,garden_tools,484,26577.22
3,389d119b48cf3043d311335e499d9c6b,garden_tools,392,21440.59
4,368c6c730842d78016ad823897a372db,garden_tools,388,21056.80
5,53759a2ecddad2bb87a079a1f1519f73,garden_tools,373,20387.20
6,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,343,47214.51
7,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,37683.42
8,154e7e31ebfa092203795c972e5804a6,health_beauty,281,6325.19
9,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,41082.60


In [94]:
query = """
SELECT

COALESCE(ct.product_category_name_english,
p.product_category_name) AS Category,

ROUND(AVG(oi.price),2)
AS Avg_Product_Price

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY Category

ORDER BY Avg_Product_Price DESC;
"""

pd.read_sql(query, conn)

,Category,Avg_Product_Price
0,computers,1098.34
1,small_appliances_home_oven_and_coffee,624.29
2,home_appliances_2,476.12
3,agro_industry_and_commerce,342.12
4,musical_instruments,281.62
...,...,...
69,food_drink,54.60
70,cds_dvds_musicals,52.14
71,diapers_and_hygiene,40.19
72,flowers,33.64


In [95]:
query = """
SELECT

payment_type,

COUNT(*) AS Payments,

ROUND(SUM(payment_value),2)
AS Total_Value,

ROUND(AVG(payment_value),2)
AS Avg_Payment

FROM payments

GROUP BY payment_type

ORDER BY Total_Value DESC;
"""

pd.read_sql(query, conn)

,payment_type,Payments,Total_Value,Avg_Payment
0,credit_card,76795,12542084.19,163.32
1,boleto,19784,2869361.27,145.03
2,voucher,5775,379436.87,65.70
3,debit_card,1529,217989.79,142.57
4,not_defined,3,0.00,0.00


In [96]:
query = """
SELECT

payment_installments,

COUNT(*) AS Transactions,

ROUND(AVG(payment_value),2)
AS Avg_Value

FROM payments

GROUP BY payment_installments

ORDER BY payment_installments;
"""

pd.read_sql(query, conn)

,payment_installments,Transactions,Avg_Value
0,0,2,94.31
1,1,52546,112.42
2,2,12413,127.23
3,3,10461,142.54
4,4,7098,163.98
5,5,5239,183.47
6,6,3920,209.85
7,7,1626,187.67
8,8,4268,307.74
9,9,644,203.44


In [97]:
query = """
SELECT

oi.seller_id,

COUNT(DISTINCT oi.order_id)
AS Orders,

ROUND(SUM(oi.price),2)
AS Revenue,

ROUND(AVG(oi.price),2)
AS Avg_Order_Value

FROM order_items oi

GROUP BY oi.seller_id

ORDER BY Revenue DESC

LIMIT 20;
"""

pd.read_sql(query, conn)

,seller_id,Orders,Revenue,Avg_Order_Value
0,4869f7a5dfa277a7dca6462dcf3b52b2,1132,229472.63,198.51
1,53243585a1d6dc2643021fd1853d8905,358,222776.05,543.36
2,4a3ca9315b744ce9f8e9374361493884,1806,200472.92,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,585,194042.03,331.13
4,7c67e1448b00f6e969d365cea6b010ab,982,187923.89,137.77
5,7e93a43ef30c4f03f38b393420bc753a,336,176431.87,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,1314,160236.57,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,1160,141745.53,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,915,138968.55,97.32
9,955fee9216a65b617aa5c0531780ce60,1287,135171.70,90.17


In [98]:
query = """
SELECT

COALESCE(ct.product_category_name_english,
p.product_category_name) AS Category,

ROUND(AVG(r.review_score),2)
AS Avg_Rating,

COUNT(r.review_id)
AS Reviews

FROM reviews r

JOIN orders o
ON r.order_id = o.order_id

JOIN order_items oi
ON o.order_id = oi.order_id

JOIN products p
ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY Category

HAVING Reviews > 100

ORDER BY Avg_Rating DESC;
"""

pd.read_sql(query, conn)

,Category,Avg_Rating,Reviews
0,books_general_interest,4.48,539
1,books_technical,4.38,265
2,food_drink,4.34,277
3,luggage_accessories,4.33,1082
4,fashion_shoes,4.27,258
5,food,4.22,494
6,stationery,4.21,2489
7,home_appliances,4.21,796
8,computers,4.21,198
9,pet_shop,4.20,1931


In [99]:
query = """
SELECT

customer_unique_id,

Total_Spent,

RANK() OVER(
ORDER BY Total_Spent DESC
) AS Customer_Rank

FROM(

SELECT

c.customer_unique_id,

ROUND(SUM(oi.price),2)
AS Total_Spent

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id

)

LIMIT 10;
"""

pd.read_sql(query,conn)

,customer_unique_id,Total_Spent,Customer_Rank
0,0a0a92112bd4c708ca5fde585afaa872,13440.0,1
1,da122df9eeddfedc1dc1f5349a1a690c,7388.0,2
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.0,3
3,dc4802a71eae9be1dd28f5d788ceb526,6735.0,4
4,459bef486812aa25204be022145caa62,6729.0,5
5,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0,6
6,4007669dec559734d6f53e029e360987,5934.6,7
7,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0,8
8,48e1ac109decbb87765a3eade6854098,4590.0,9
9,a229eba70ec1c2abef51f04987deb7a5,4400.0,10


In [100]:
query="""
SELECT

seller_id,

Revenue,

DENSE_RANK() OVER(
ORDER BY Revenue DESC
) AS Seller_Rank

FROM(

SELECT

seller_id,

ROUND(SUM(price),2)
AS Revenue

FROM order_items

GROUP BY seller_id

);
"""

pd.read_sql(query,conn)

,seller_id,Revenue,Seller_Rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1
1,53243585a1d6dc2643021fd1853d8905,222776.05,2
2,4a3ca9315b744ce9f8e9374361493884,200472.92,3
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,4
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,5
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,2762
3091,702835e4b785b67a084280efca355756,7.60,2763
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,2764
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,2765


In [101]:
query="""
WITH ProductRevenue AS(

SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

p.product_id,

ROUND(SUM(oi.price),2)
AS Revenue

FROM order_items oi

JOIN products p
ON oi.product_id=p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name=
ct.product_category_name

GROUP BY Category,p.product_id

)

SELECT *

FROM(

SELECT *,

ROW_NUMBER() OVER(

PARTITION BY Category

ORDER BY Revenue DESC

) AS rn

FROM ProductRevenue

)

WHERE rn=1;
"""

pd.read_sql(query,conn)

,Category,product_id,Revenue,rn
0,Unknown,5a848e4ab52fd5445cdc07aab1c40e48,24229.03,1
1,agro_industry_and_commerce,11250b0d4b709fee92441c5f34122aed,9111.00,1
2,air_conditioning,12485f9cdebb6ca179826ede539554ad,3899.91,1
3,art,4fe644d766c7566dbc46fb851363cb3b,10803.72,1
4,arts_and_craftmanship,b9976e9c22fb1540bd71d1bcd2989475,641.45,1
...,...,...,...,...
69,stationery,5411e9269501a870cabf632f05655131,9153.00,1
70,tablets_printing_image,6bbe55cf8f85c87b6eebb775a53402f4,2523.56,1
71,telephony,e7cc48a9daff5436f63d3aad9426f28b,16216.00,1
72,toys,dc404a1496a08f9f5540c8b5d4b92925,11961.40,1


In [102]:
query="""
WITH MonthlyRevenue AS(

SELECT

strftime('%Y-%m',
o.order_purchase_timestamp)
AS Month,

SUM(oi.price)
AS Revenue

FROM orders o

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY Month

)

SELECT

Month,

ROUND(Revenue,2)
AS Revenue,

ROUND(

SUM(Revenue)
OVER(
ORDER BY Month
),2)

AS Running_Revenue

FROM MonthlyRevenue;
"""

pd.read_sql(query,conn)

,Month,Revenue,Running_Revenue
0,2016-09,267.36,267.36
1,2016-10,49507.66,49775.02
2,2016-12,10.90,49785.92
3,2017-01,120312.87,170098.79
4,2017-02,247303.02,417401.81
5,2017-03,374229.40,791631.21
6,2017-04,359229.62,1150860.83
7,2017-05,505185.88,1656046.71
8,2017-06,432574.42,2088621.13
9,2017-07,494737.64,2583358.77


In [103]:
query="""
WITH MonthlyRevenue AS(

SELECT

strftime('%Y-%m',
o.order_purchase_timestamp)
AS Month,

SUM(oi.price)
AS Revenue

FROM orders o

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY Month

)

SELECT

Month,

ROUND(Revenue,2)
AS Revenue,

ROUND(

LAG(Revenue)
OVER(
ORDER BY Month
),2)

AS Previous_Month,

ROUND(

(
Revenue-

LAG(Revenue)
OVER(
ORDER BY Month
)

)

/

LAG(Revenue)
OVER(
ORDER BY Month
)

*100,2)

AS Growth_Percentage

FROM MonthlyRevenue;
"""

pd.read_sql(query,conn)

,Month,Revenue,Previous_Month,Growth_Percentage
0,2016-09,267.36,NaN,NaN
1,2016-10,49507.66,267.36,18417.23
2,2016-12,10.90,49507.66,-99.98
3,2017-01,120312.87,10.90,1103687.80
4,2017-02,247303.02,120312.87,105.55
5,2017-03,374229.40,247303.02,51.32
6,2017-04,359229.62,374229.40,-4.01
7,2017-05,505185.88,359229.62,40.63
8,2017-06,432574.42,505185.88,-14.37
9,2017-07,494737.64,432574.42,14.37


In [104]:
query="""
WITH SellerRevenue AS(

SELECT

seller_id,

SUM(price)
AS Revenue

FROM order_items

GROUP BY seller_id

)

SELECT

seller_id,

ROUND(Revenue,2)
AS Revenue,

ROUND(

SUM(Revenue)
OVER(

ORDER BY Revenue DESC

)

,2)

AS Running_Revenue

FROM SellerRevenue

ORDER BY Revenue DESC;
"""

pd.read_sql(query,conn)

,seller_id,Revenue,Running_Revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05,452248.68
2,4a3ca9315b744ce9f8e9374361493884,200472.92,652721.60
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,846763.63
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,1034687.52
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,13591619.20
3091,702835e4b785b67a084280efca355756,7.60,13591626.80
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,13591633.70
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,13591640.20


In [105]:
query="""
SELECT

c.customer_unique_id,

COUNT(DISTINCT o.order_id)
AS Orders,

ROUND(SUM(oi.price),2)
AS Lifetime_Value

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id

ORDER BY Lifetime_Value DESC

LIMIT 20;
"""

pd.read_sql(query,conn)

,customer_unique_id,Orders,Lifetime_Value
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00
4,459bef486812aa25204be022145caa62,1,6729.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00
6,4007669dec559734d6f53e029e360987,1,5934.60
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00
8,48e1ac109decbb87765a3eade6854098,1,4590.00
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.00


In [106]:
query="""
WITH CustomerOrders AS(

SELECT

c.customer_unique_id,

COUNT(DISTINCT o.order_id)
AS Orders

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

GROUP BY c.customer_unique_id

)

SELECT

CASE

WHEN Orders=1
THEN 'One-Time'

ELSE 'Repeat'

END
AS Customer_Type,

COUNT(*)
AS Customers

FROM CustomerOrders

GROUP BY Customer_Type;
"""

pd.read_sql(query,conn)

,Customer_Type,Customers
0,One-Time,92078
1,Repeat,2910


In [107]:
query = """
SELECT

c.customer_unique_id,

MAX(date(o.order_purchase_timestamp))
AS Last_Order,

CAST(

julianday(
MAX(date(o.order_purchase_timestamp))
)

-

julianday(
MIN(date(o.order_purchase_timestamp))
)

AS INTEGER)

AS Customer_Age,

COUNT(DISTINCT o.order_id)
AS Frequency,

ROUND(SUM(oi.price),2)
AS Monetary

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id;
"""

pd.read_sql(query,conn)

,customer_unique_id,Last_Order,Customer_Age,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10,0,1,129.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07,0,1,18.90
2,0000f46a3911fa3c0805444483337064,2017-03-10,0,1,69.00
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12,0,1,25.99
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14,0,1,180.00
...,...,...,...,...,...
94525,fffcf5a5ff07b0908bd4e2dbc735a684,2017-06-08,0,1,1570.00
94526,fffea47cd6d3cc0a88bd621562a9d061,2017-12-10,0,1,64.89
94527,ffff371b4d645b6ecea244b27531430a,2017-02-07,0,1,89.90
94528,ffff5962728ec6157033ef9805bacc48,2018-05-02,0,1,115.00


In [108]:
query="""
SELECT

c.customer_unique_id,

COUNT(DISTINCT o.order_id)
AS Orders,

ROUND(SUM(oi.price),2)
AS Lifetime_Value,

ROUND(AVG(oi.price),2)
AS Avg_Order_Value

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id

ORDER BY Lifetime_Value DESC

LIMIT 20;
"""

pd.read_sql(query,conn)

,customer_unique_id,Orders,Lifetime_Value,Avg_Order_Value
0,0a0a92112bd4c708ca5fde585afaa872,1,13440.00,1680.00
1,da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,3694.00
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7160.00,1790.00
3,dc4802a71eae9be1dd28f5d788ceb526,1,6735.00,6735.00
4,459bef486812aa25204be022145caa62,1,6729.00,6729.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6499.00,6499.00
6,4007669dec559734d6f53e029e360987,1,5934.60,989.10
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4690.00,4690.00
8,48e1ac109decbb87765a3eade6854098,1,4590.00,4590.00
9,a229eba70ec1c2abef51f04987deb7a5,1,4400.00,2200.00


In [109]:
query="""
SELECT

customer_unique_id,

Lifetime_Value

FROM(

SELECT

c.customer_unique_id,

ROUND(SUM(oi.price),2)
AS Lifetime_Value

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id

)

WHERE Lifetime_Value>1000

ORDER BY Lifetime_Value DESC;
"""

pd.read_sql(query,conn)

,customer_unique_id,Lifetime_Value
0,0a0a92112bd4c708ca5fde585afaa872,13440.00
1,da122df9eeddfedc1dc1f5349a1a690c,7388.00
2,763c8b1c9c68a0229c42c9fc6f662b93,7160.00
3,dc4802a71eae9be1dd28f5d788ceb526,6735.00
4,459bef486812aa25204be022145caa62,6729.00
...,...,...
957,6bda5f629d7f63fbd492f48da584b8c5,1004.99
958,769151568e474636f23211792b0e32c1,1004.99
959,db3d2c340a500126dec4308cee9135ae,1004.99
960,7b50def89c51f42400b796d7787ed658,1001.00


In [110]:
query="""
WITH CustomerRevenue AS(

SELECT

c.customer_unique_id,

ROUND(SUM(oi.price),2)
AS Revenue

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY c.customer_unique_id

)

SELECT

CASE

WHEN Revenue>=1000
THEN 'VIP'

WHEN Revenue>=500
THEN 'Premium'

WHEN Revenue>=200
THEN 'Regular'

ELSE 'Low Value'

END
AS Segment,

COUNT(*) AS Customers,

ROUND(AVG(Revenue),2)
AS Avg_Revenue

FROM CustomerRevenue

GROUP BY Segment

ORDER BY Avg_Revenue DESC;
"""

pd.read_sql(query,conn)

,Segment,Customers,Avg_Revenue
0,VIP,965,1620.86
1,Premium,2753,687.78
2,Regular,11659,298.90
3,Low Value,79153,82.06


In [111]:
query="""
SELECT

ROUND(

SUM(price)

/

COUNT(DISTINCT customer_id)

,2)

AS Revenue_Per_Customer

FROM orders o

JOIN order_items oi
ON o.order_id=oi.order_id;
"""

pd.read_sql(query,conn)

,Revenue_Per_Customer
0,137.5


In [112]:
query="""
SELECT

strftime('%Y-%m',

MIN(order_purchase_timestamp)

)

AS Cohort,

COUNT(DISTINCT customer_id)
AS Customers

FROM orders

GROUP BY customer_id

ORDER BY Cohort;
"""

pd.read_sql(query,conn)

,Cohort,Customers
0,2016-09,1
1,2016-09,1
2,2016-09,1
3,2016-09,1
4,2016-10,1
...,...,...
98227,2018-08,1
98228,2018-08,1
98229,2018-08,1
98230,2018-08,1


In [113]:
query="""
SELECT

strftime('%Y-%m',
order_purchase_timestamp)

AS Month,

COUNT(DISTINCT customer_id)
AS Active_Customers

FROM orders

GROUP BY Month

ORDER BY Month;
"""

pd.read_sql(query,conn)

,Month,Active_Customers
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,799
4,2017-02,1780
5,2017-03,2679
6,2017-04,2399
7,2017-05,3687
8,2017-06,3236
9,2017-07,4001


In [114]:
query="""
WITH CustomerMonth AS(

SELECT

customer_id,

strftime('%Y-%m',
order_purchase_timestamp)
AS Month

FROM orders

)

SELECT

Month,

COUNT(customer_id)
AS Repeat_Customers

FROM CustomerMonth

GROUP BY Month

HAVING COUNT(customer_id)>1

ORDER BY Month;
"""

pd.read_sql(query,conn)

,Month,Repeat_Customers
0,2016-09,4
1,2016-10,324
2,2017-01,799
3,2017-02,1780
4,2017-03,2679
5,2017-04,2399
6,2017-05,3687
7,2017-06,3236
8,2017-07,4001
9,2017-08,4302


In [115]:
query="""
SELECT

COUNT(DISTINCT o.order_id)
AS Total_Orders,

COUNT(DISTINCT c.customer_unique_id)
AS Customers,

COUNT(DISTINCT oi.product_id)
AS Products,

COUNT(DISTINCT oi.seller_id)
AS Sellers,

ROUND(SUM(oi.price),2)
AS Revenue,

ROUND(AVG(oi.price),2)
AS Average_Order_Value,

ROUND(AVG(r.review_score),2)
AS Avg_Rating

FROM orders o

JOIN customers c
ON o.customer_id=c.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

LEFT JOIN reviews r
ON o.order_id=r.order_id;
"""

pd.read_sql(query,conn)

,Total_Orders,Customers,Products,Sellers,Revenue,Average_Order_Value,Avg_Rating
0,97733,94530,32671,3058,13497586.4,120.21,4.05


In [116]:
query = """
WITH CustomerRevenue AS (

SELECT

c.customer_state,

c.customer_unique_id,

ROUND(SUM(oi.price),2)
AS Revenue

FROM customers c

JOIN orders o
ON c.customer_id=o.customer_id

JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY
c.customer_state,
c.customer_unique_id

)

SELECT *

FROM(

SELECT *,

ROW_NUMBER() OVER(

PARTITION BY customer_state

ORDER BY Revenue DESC

) AS Rank

FROM CustomerRevenue

)

WHERE Rank<=10

ORDER BY customer_state,Rank;
"""

pd.read_sql(query,conn)

,customer_state,customer_unique_id,Revenue,Rank
0,AC,62a459e5629b03dd73134964df732077,1200.00,1
1,AC,086d6b5b5ba195a91aa0a6ec8e75d1a4,961.60,2
2,AC,3947ca729a860c522a64a49d762baada,839.99,3
3,AC,3e5c928acf49c4b95e57af1f350d3493,809.10,4
4,AC,28989ef45087c96e5a4346e88216c2ba,589.60,5
...,...,...,...,...
265,TO,b6a70553f73cde17709a32cb714a40eb,987.00,6
266,TO,18ac369326ec0aafe3ae0e397b14102b,874.00,7
267,TO,4ad0a855f1b51a52b45d3a3150fd1ebf,859.00,8
268,TO,1b8fa1f83767978408ddfdd4105d31e4,810.00,9


In [117]:
query="""
WITH SellerRevenue AS(

SELECT

seller_id,

SUM(price)
AS Revenue

FROM order_items

GROUP BY seller_id

)

SELECT

seller_id,

ROUND(Revenue,2)
AS Revenue,

ROUND(

Revenue*100.0/

SUM(Revenue)
OVER()

,2)

AS Market_Share

FROM SellerRevenue

ORDER BY Revenue DESC;
"""

pd.read_sql(query,conn)

,seller_id,Revenue,Market_Share
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1.69
1,53243585a1d6dc2643021fd1853d8905,222776.05,1.64
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1.47
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,1.43
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,1.38
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,0.00
3091,702835e4b785b67a084280efca355756,7.60,0.00
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,0.00
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,0.00


In [118]:
query="""
WITH CategoryRevenue AS(

SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

SUM(oi.price)
AS Revenue

FROM order_items oi

JOIN products p
ON oi.product_id=p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name=
ct.product_category_name

GROUP BY Category

)

SELECT

Category,

ROUND(Revenue,2)
AS Revenue,

ROUND(

Revenue*100.0/

SUM(Revenue)
OVER()

,2)

AS Contribution

FROM CategoryRevenue

ORDER BY Revenue DESC;
"""

pd.read_sql(query,conn)

,Category,Revenue,Contribution
0,health_beauty,1258681.34,9.26
1,watches_gifts,1205005.68,8.87
2,bed_bath_table,1036988.68,7.63
3,sports_leisure,988048.97,7.27
4,computers_accessories,911954.32,6.71
...,...,...,...
69,flowers,1110.04,0.01
70,home_comfort_2,760.27,0.01
71,cds_dvds_musicals,730.00,0.01
72,fashion_childrens_clothes,569.85,0.00


In [119]:
query="""
WITH SellerRevenue AS(

SELECT

seller_id,

SUM(price)
AS Revenue

FROM order_items

GROUP BY seller_id

),

RunningRevenue AS(

SELECT

seller_id,

Revenue,

SUM(Revenue)

OVER(

ORDER BY Revenue DESC

)

AS RunningRevenue,

SUM(Revenue)
OVER()

AS TotalRevenue

FROM SellerRevenue

)

SELECT

seller_id,

ROUND(Revenue,2)
AS Revenue,

ROUND(

RunningRevenue*100.0/

TotalRevenue

,2)

AS Cumulative_Percentage

FROM RunningRevenue

ORDER BY Revenue DESC;
"""

pd.read_sql(query,conn)

,seller_id,Revenue,Cumulative_Percentage
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1.69
1,53243585a1d6dc2643021fd1853d8905,222776.05,3.33
2,4a3ca9315b744ce9f8e9374361493884,200472.92,4.80
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,6.23
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,7.61
...,...,...,...
3090,34aefe746cd81b7f3b23253ea28bef39,8.00,100.00
3091,702835e4b785b67a084280efca355756,7.60,100.00
3092,1fa2d3def6adfa70e58c276bb64fe5bb,6.90,100.00
3093,77128dec4bec4878c37ab7d6169d6f26,6.50,100.00


In [120]:
query="""
SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

ROUND(
AVG(r.review_score),2)
AS Avg_Rating,

COUNT(r.review_id)
AS Reviews

FROM reviews r

JOIN orders o
ON r.order_id=o.order_id

JOIN order_items oi
ON o.order_id=oi.order_id

JOIN products p
ON oi.product_id=p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name=
ct.product_category_name

GROUP BY Category

HAVING Reviews>100

ORDER BY Avg_Rating DESC;
"""

pd.read_sql(query,conn)

,Category,Avg_Rating,Reviews
0,books_general_interest,4.48,539
1,books_technical,4.38,265
2,food_drink,4.34,277
3,luggage_accessories,4.33,1082
4,fashion_shoes,4.27,258
5,food,4.22,494
6,stationery,4.21,2489
7,home_appliances,4.21,796
8,computers,4.21,198
9,pet_shop,4.20,1931


In [121]:
query="""
SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

ROUND(
AVG(r.review_score),2)
AS Avg_Rating,

COUNT(r.review_id)
AS Reviews

FROM reviews r

JOIN orders o
ON r.order_id=o.order_id

JOIN order_items oi
ON o.order_id=oi.order_id

JOIN products p
ON oi.product_id=p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name=
ct.product_category_name

GROUP BY Category

HAVING Reviews>100

ORDER BY Avg_Rating ASC;
"""

pd.read_sql(query,conn)

,Category,Avg_Rating,Reviews
0,office_furniture,3.50,1680
1,fashion_male_clothing,3.63,130
2,fixed_telephony,3.69,261
3,audio,3.82,360
4,home_confort,3.84,433
5,Unknown,3.88,1574
6,bed_bath_table,3.90,11081
7,construction_tools_safety,3.90,189
8,furniture_decor,3.92,8264
9,furniture_living_room,3.92,500


In [122]:
query="""
SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

ROUND(
SUM(oi.price),2)
AS Revenue,

ROUND(
AVG(r.review_score),2)
AS Rating

FROM reviews r

JOIN orders o
ON r.order_id=o.order_id

JOIN order_items oi
ON o.order_id=oi.order_id

JOIN products p
ON oi.product_id=p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name=
ct.product_category_name

GROUP BY Category

ORDER BY Revenue DESC;
"""

pd.read_sql(query,conn)

,Category,Revenue,Rating
0,health_beauty,1241307.22,4.16
1,watches_gifts,1182769.48,4.05
2,bed_bath_table,1035380.33,3.90
3,sports_leisure,974088.17,4.13
4,computers_accessories,906442.42,3.95
...,...,...,...
69,flowers,1000.24,4.42
70,cds_dvds_musicals,730.00,4.64
71,home_comfort_2,721.57,3.63
72,fashion_childrens_clothes,569.85,4.50


In [123]:
query = """
SELECT

order_status,

COUNT(*) AS Orders,

ROUND(
AVG(
julianday(order_delivered_customer_date) -
julianday(order_estimated_delivery_date)
),2) AS Avg_Delay_Days

FROM orders

WHERE order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL

GROUP BY order_status

ORDER BY Avg_Delay_Days DESC;
"""

pd.read_sql(query, conn)

,order_status,Orders,Avg_Delay_Days
0,approved,2,403.94
1,processing,212,204.41
2,unavailable,423,202.21
3,invoiced,205,194.03
4,canceled,311,188.64
5,shipped,604,173.98
6,created,4,153.36
7,delivered,96471,-11.17


In [124]:
query = """
SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

ROUND(AVG(oi.freight_value),2)
AS Avg_Freight,

ROUND(SUM(oi.freight_value),2)
AS Total_Freight

FROM order_items oi

JOIN products p
ON oi.product_id = p.product_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY Category

ORDER BY Total_Freight DESC;
"""

pd.read_sql(query, conn)

,Category,Avg_Freight,Total_Freight
0,bed_bath_table,18.42,204693.04
1,health_beauty,18.88,182566.73
2,furniture_decor,20.73,172749.30
3,sports_leisure,19.51,168607.51
4,computers_accessories,18.82,147318.08
...,...,...,...
69,portateis_cozinha_e_preparadores_de_alimentos,20.65,309.76
70,cds_dvds_musicals,16.07,224.99
71,pc_gamer,14.84,133.57
72,fashion_childrens_clothes,11.94,95.51


In [125]:
query = """
SELECT

o.order_id,

c.customer_state,

ROUND(SUM(oi.price),2) AS Order_Value,

COUNT(oi.product_id) AS Items

FROM orders o

JOIN customers c
ON o.customer_id = c.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY o.order_id

ORDER BY Order_Value DESC

LIMIT 10;
"""

pd.read_sql(query, conn)

,order_id,customer_state,Order_Value,Items
0,03caa2c082116e1d31e67e9ae3700499,RJ,13440.0,8
1,736e1922ae60d0d6a89247b851902527,ES,7160.0,4
2,0812eb902a67711a1cb742b3cdaa65ae,MS,6735.0,1
3,fefacc66af859508bf1a7934eab1e97f,ES,6729.0,1
4,f5136e38d1a14a4dbd87dff67da82701,SP,6499.0,1
5,2cc9089445046817a7539d90805e6e5a,MG,5934.6,6
6,a96610ab360d42a2e5335a3998b4718a,RJ,4799.0,1
7,199af31afc78c699f0dbf71fb178d4d4,SP,4690.0,1
8,8dbc85d1447242f3b127dda390d56e19,PB,4590.0,1
9,d2f270487125ddc41fd134c4003ad1d7,RJ,4400.0,2


In [126]:
query = """
SELECT

c.customer_state,

ROUND(AVG(r.review_score),2)
AS Avg_Rating,

COUNT(r.review_id)
AS Reviews

FROM reviews r

JOIN orders o
ON r.order_id = o.order_id

JOIN customers c
ON o.customer_id = c.customer_id

GROUP BY c.customer_state

HAVING Reviews > 100

ORDER BY Avg_Rating DESC;
"""

pd.read_sql(query, conn)

,customer_state,Avg_Rating,Reviews
0,SP,4.20,41166
1,PR,4.20,4996
2,AM,4.18,147
3,RS,4.15,5446
4,MG,4.15,11532
5,MS,4.14,718
6,RN,4.13,476
7,MT,4.13,893
8,TO,4.11,277
9,SC,4.09,3594


In [127]:
query = """
SELECT

COALESCE(
ct.product_category_name_english,
p.product_category_name
) AS Category,

ROUND(SUM(oi.price),2) AS Revenue,

ROUND(AVG(r.review_score),2) AS Rating

FROM products p

JOIN order_items oi
ON p.product_id = oi.product_id

JOIN orders o
ON oi.order_id = o.order_id

LEFT JOIN reviews r
ON o.order_id = r.order_id

LEFT JOIN category_translation ct
ON p.product_category_name = ct.product_category_name

GROUP BY Category

HAVING Revenue > 100000

ORDER BY Rating ASC;
"""

pd.read_sql(query, conn)
query = """
SELECT

oi.seller_id,

COUNT(DISTINCT oi.order_id) AS Orders,

ROUND(SUM(oi.price),2) AS Revenue,

ROUND(AVG(r.review_score),2) AS Avg_Rating,

ROUND(AVG(oi.freight_value),2) AS Avg_Freight

FROM order_items oi

JOIN orders o
ON oi.order_id = o.order_id

LEFT JOIN reviews r
ON o.order_id = r.order_id

GROUP BY oi.seller_id

HAVING Orders >= 20

ORDER BY Revenue DESC;
"""

pd.read_sql(query, conn)

,seller_id,Orders,Revenue,Avg_Rating,Avg_Freight
0,4869f7a5dfa277a7dca6462dcf3b52b2,1129,228414.83,4.13,17.43
1,53243585a1d6dc2643021fd1853d8905,353,221497.05,4.10,32.06
2,4a3ca9315b744ce9f8e9374361493884,1793,201724.02,3.81,17.59
3,fa1c13f2614d7b5c4749cbc52fecda94,580,192716.43,4.36,17.13
4,7c67e1448b00f6e969d365cea6b010ab,980,188967.69,3.35,37.77
...,...,...,...,...,...
806,6b15924333bd1a741595fe981ea04822,21,614.32,4.67,11.44
807,0d33a55da925bbf1ff02af5f6059fc7f,22,575.30,4.52,15.80
808,916748bc99315c2d202898ae58b1617e,25,369.40,4.34,11.78
809,48efc9d94a9834137efd9ea76b065a38,33,345.60,5.00,9.42
